# Tech Addiction Prediction: HistGradientBoosting Model
In this notebook, we build a robust tree-based model using Scikit-Learn's `HistGradientBoostingClassifier`. 
We will implement a 5-Fold Stratified Cross Validation, directly optimizing the model for the competition metric (`ROC-AUC`), and train a final model on the entire dataset to make our probability predictions.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score


## 1. Data Loading
We load the data from the standard Kaggle input directory.


In [ ]:
TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e8/train.csv'
TEST_PATH = '/kaggle/input/competitions/playground-series-s6e8/test.csv'
SUBMISSION_PATH = '/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv'

print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUBMISSION_PATH)

X = train_df.drop(['id', 'addicted_label'], axis=1)
y = train_df['addicted_label'].values

print(f"Train features shape: {X.shape}")
print(f"Test features shape: {test_df.drop(['id'], axis=1).shape}")


## 2. Preprocessing Pipeline
Trees handle missing values natively, so we do not need imputers for numerical values or standard scalers.


In [ ]:
categorical_nominal = ['gender', 'academic_work_impact']
categorical_ordinal = ['stress_level']

nominal_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

ordinal_transformer = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories=[['Low', 'Medium', 'High']], handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('nom', nominal_transformer, categorical_nominal),
        ('ord', ordinal_transformer, categorical_ordinal)
    ],
    remainder='passthrough'
)


## 3. Stratified 5-Fold Cross Validation
We evaluate the model using 5 folds to generate Out-Of-Fold (OOF) predictions. We configure the model to use internal early stopping by monitoring the `roc_auc` scoring metric!


In [ ]:
print("Starting 5-Fold Stratified Cross-Validation for Tree Model...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"Training Fold {fold + 1}/5...")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', HistGradientBoostingClassifier(
            random_state=42, 
            max_iter=1000, 
            early_stopping=True, 
            scoring='roc_auc', 
            validation_fraction=0.1
        ))
    ])
    
    pipeline.fit(X_train, y_train)
    oof_preds[val_idx] = pipeline.predict_proba(X_val)[:, 1]

print("\nCross-Validation complete!")


### Metric Evaluation


In [ ]:
print("Evaluating OOF predictions...")
roc_auc = roc_auc_score(y, oof_preds)
pr_auc = average_precision_score(y, oof_preds)

print("-" * 30)
print("Tree Model (5-Fold OOF) Performance:")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")
print("-" * 30)


## 4. Train Final Model on Full Dataset
To maximize our predictive power on the test set, we retrain using **100%** of our training data.


In [ ]:
print("Training final model on full dataset...")
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(
        random_state=42, 
        max_iter=1000, 
        early_stopping=True, 
        scoring='roc_auc', 
        validation_fraction=0.1
    ))
])

final_pipeline.fit(X, y)
print("Final model trained!")


## 5. Inference and Submission
We predict the raw probabilities for the unseen test set and write directly to `submission.csv` as required by the ROC-AUC evaluation metric.


In [ ]:
# Get test features
X_test = test_df.drop(['id'], axis=1)

# Predict probabilities
test_preds_proba = final_pipeline.predict_proba(X_test)[:, 1]

# Create submission (using probabilities directly!)
submission = pd.DataFrame({
    'id': test_df['id'],
    'addicted_label': test_preds_proba
})

submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")
display(submission.head())
